# 2. Supervised Modeling: Regression and Classification
This notebook contains MLflow-tracked experiments for regression and multi-class classification tasks, including cross-validation, hyperparameter tuning, and threshold calibration.

## 1. Import Required Libraries
We import all necessary libraries for regression, classification, evaluation, and experiment tracking.

In [ ]:
# Data and modeling
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, confusion_matrix, classification_report, f1_score, precision_score, recall_score, log_loss
from sklearn.pipeline import Pipeline
import joblib
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import os

## 2. Load Data and Preprocessing Pipeline
We load the cleaned dataset and the preprocessing pipeline from the previous notebook.

In [ ]:
# Load cleaned data (repeat cleaning steps for reproducibility)
df = pd.read_csv('https://huggingface.co/datasets/millat/e-commerce-orders/resolve/main/ecommerce_orders.csv')
drop_cols = ['order_id', 'customer_id', 'product_id', 'shipping_address', 'billing_address']
df_clean = df.drop(columns=drop_cols)
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'])
df_clean['shipping_date'] = pd.to_datetime(df_clean['shipping_date'])
for col in ['order_date', 'shipping_date']:
    df_clean[f'{col}_year'] = df_clean[col].dt.year
    df_clean[f'{col}_month'] = df_clean[col].dt.month
    df_clean[f'{col}_day'] = df_clean[col].dt.day
    df_clean[f'{col}_hour'] = df_clean[col].dt.hour
df_clean['days_to_ship'] = (df_clean['shipping_date'] - df_clean['order_date']).dt.days
df_clean = df_clean.drop(columns=['order_date', 'shipping_date'])
df_clean = df_clean.fillna({'days_to_ship': df_clean['days_to_ship'].median()})
df_clean = df_clean.dropna()

# Load preprocessing pipeline
preprocessor = joblib.load('../artifacts/preprocessing_pipeline.joblib')

## 3. Regression Modeling: Predicting Order Price
We will build and evaluate a multiple linear regression model to predict the price of an order.

In [ ]:
# Prepare features and target for regression
categorical_targets = ['customer_segment', 'delivery_status']
X = df_clean.drop(columns=categorical_targets + ['price'])
y = df_clean['price']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build pipeline
reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Train model
reg_pipeline.fit(X_train, y_train)

# Predict
y_pred = reg_pipeline.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
print(f"Test MSE: {mse:.2f}")
print(f"Test MAE: {mae:.2f}")

### Cross-Validation and Hyperparameter Analysis
We use k-fold cross-validation to assess model reliability and discuss the impact of key parameters.

In [ ]:
# k-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_mse = cross_val_score(reg_pipeline, X, y, cv=kf, scoring='neg_mean_squared_error')
cv_mae = cross_val_score(reg_pipeline, X, y, cv=kf, scoring='neg_mean_absolute_error')
print(f"CV MSE: {-cv_mse.mean():.2f} (+/- {cv_mse.std():.2f})")
print(f"CV MAE: {-cv_mae.mean():.2f} (+/- {cv_mae.std():.2f})")

### MLflow Tracking for Regression
We log the regression experiment, parameters, and metrics using MLflow.

In [ ]:
# MLflow tracking for regression
mlflow.set_experiment('AuraCart Regression')
with mlflow.start_run(run_name='LinearRegression'):
    mlflow.sklearn.autolog()
    reg_pipeline.fit(X_train, y_train)
    y_pred = reg_pipeline.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mlflow.log_metric('test_mse', mse)
    mlflow.log_metric('test_mae', mae)
    mlflow.sklearn.log_model(reg_pipeline, 'regression_model')
    print(f"Logged regression run with MSE: {mse:.2f}, MAE: {mae:.2f}")

## 4. Multi-class Classification: Predicting Customer Segment
We will build and evaluate a multinomial logistic regression (softmax regression) model to predict the customer segment.

In [ ]:
# Prepare features and target for classification
X_cls = df_clean.drop(columns=['customer_segment', 'delivery_status', 'price'])
y_cls = df_clean['customer_segment']

# Split data
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

# Build pipeline
cls_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=500, class_weight='balanced'))
])

# Train model
cls_pipeline.fit(Xc_train, yc_train)

# Predict
yc_pred = cls_pipeline.predict(Xc_test)
yc_proba = cls_pipeline.predict_proba(Xc_test)

# Evaluate
print(classification_report(yc_test, yc_pred))

### Confusion Matrix and Class-wise Metrics
We analyze the confusion matrix and calculate precision, recall, and F1-score for each class.

In [ ]:
# Confusion matrix
cm = confusion_matrix(yc_test, yc_pred, labels=cls_pipeline.classes_)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cls_pipeline.classes_, yticklabels=cls_pipeline.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Customer Segment')
plt.show()

# Class-wise metrics
print('Precision:', precision_score(yc_test, yc_pred, average=None, labels=cls_pipeline.classes_))
print('Recall:', recall_score(yc_test, yc_pred, average=None, labels=cls_pipeline.classes_))
print('F1-score:', f1_score(yc_test, yc_pred, average=None, labels=cls_pipeline.classes_))

### Decision Threshold Analysis
We experiment with different probability thresholds to improve detection of rare classes.

In [ ]:
# Example: Adjust threshold for 'VIP' class
def threshold_predict(proba, class_idx, threshold):
    pred = np.argmax(proba, axis=1)
    pred_proba = proba[:, class_idx]
    pred[pred_proba > threshold] = class_idx
    return pred

vip_idx = list(cls_pipeline.classes_).index('VIP')
thresholds = [0.3, 0.4, 0.5, 0.6]
for t in thresholds:
    pred_thr = threshold_predict(yc_proba, vip_idx, t)
    print(f"\nThreshold {t} for VIP:")
    print(classification_report(yc_test, [cls_pipeline.classes_[i] for i in pred_thr]))

### MLflow Tracking for Classification
We log the classification experiment, parameters, and metrics using MLflow.

In [ ]:
# MLflow tracking for classification
mlflow.set_experiment('AuraCart Classification')
with mlflow.start_run(run_name='SoftmaxRegression'):
    mlflow.sklearn.autolog()
    cls_pipeline.fit(Xc_train, yc_train)
    yc_pred = cls_pipeline.predict(Xc_test)
    f1 = f1_score(yc_test, yc_pred, average='weighted')
    logloss = log_loss(yc_test, cls_pipeline.predict_proba(Xc_test), labels=cls_pipeline.classes_)
    mlflow.log_metric('test_f1', f1)
    mlflow.log_metric('test_logloss', logloss)
    mlflow.sklearn.log_model(cls_pipeline, 'classification_model')
    print(f"Logged classification run with F1: {f1:.2f}, LogLoss: {logloss:.2f}")